# 04 - Fusion hai mô hình: Pretrained Backbone + Co-Attention 5-Fold + 10-Fold

Notebook này ghép output của hai notebook 03:

- **03A pretrained/raw-audio backbone**: lưu `*_pretrained_features.npz`.
- **03B co-attention model riêng**: lưu `*_coattention_features.npz`.

Fusion vẫn chạy theo từng fold. Với mỗi fold:

- Train fusion trên feature `train` của đúng fold.
- Chọn checkpoint bằng `val`.
- Báo cáo cuối trên `test`.

Như vậy ta không lấy checkpoint đã nhìn thấy test speaker/session để tạo tín hiệu cho test.

## Mục tiêu của notebook 04

Notebook 04 là bước cuối để trả lời câu hỏi nghiên cứu:

> Nếu một pretrained raw-audio model đã fine-tune tốt và một co-attention acoustic model học tín hiệu bổ sung khác nhau, việc fusion hai nguồn này có cải thiện emotion classification và VAD regression không?

Ta không fusion bằng cách train một model trên toàn bộ dataset rồi dự đoán lại toàn bộ. Cách đó dễ làm rò rỉ test. Ở đây fusion vẫn theo fold:

```text
fold k:
  03A train trên train_k -> xuất feature train/val/test_k
  03B train trên train_k -> xuất feature train/val/test_k
  04 train fusion trên train_k
  04 chọn checkpoint bằng val_k
  04 báo cáo test_k
```

Vì vậy, mỗi mẫu test chỉ nhận tín hiệu từ các model chưa từng train trên speaker/session test đó.

## Ký hiệu của fusion model

Với một utterance \(i\):

| Ký hiệu | Ý nghĩa |
|---|---|
| \(z_i^{pre}\) | embedding từ pretrained/raw-audio model 03A |
| \(p_i^{pre}\) | xác suất emotion từ 03A |
| \(\hat{v}_i^{pre}\) | dự đoán VAD từ 03A |
| \(z_i^{co}\) | embedding từ co-attention model 03B |
| \(p_i^{co}\) | xác suất emotion từ 03B |
| \(\hat{v}_i^{co}\) | dự đoán VAD từ 03B |

Vector fusion đầu vào:

$$
x_i^{fusion}
=
[z_i^{pre}, p_i^{pre}, \hat{v}_i^{pre}, z_i^{co}, p_i^{co}, \hat{v}_i^{co}]
$$

Fusion MLP:

$$
h_i=f_{\theta}(x_i^{fusion})
$$

Hai head:

$$
\hat{p}_i=\operatorname{softmax}(W_ch_i+b_c)
$$

$$
\hat{y}_{i,VAD}=\sigma(W_rh_i+b_r)
$$

## Dữ liệu cần có

Upload hoặc đặt hai output folder/zip từ 03A và 03B vào Kaggle:

```text
output_03a_pretrained_backbone/fusion_features/*.npz
output_03b_coattention/fusion_features/*.npz
```

Notebook sẽ tự tìm các file `.npz` trong `/kaggle/input`, `/kaggle/working` và project local.
Nếu bạn upload output ở dạng `.zip`, notebook sẽ tự giải nén các file zip liên quan vào thư mục working trước khi tìm feature.

## Vì sao fusion phải chạy theo fold?

Nếu lấy một model đã train toàn bộ rồi sinh feature cho toàn bộ dataset, điểm fusion có thể bị cao giả vì test speaker/session đã gián tiếp xuất hiện trong mô hình tạo feature.

Vì vậy notebook 04 chỉ ghép các feature được tạo theo đúng fold:

- Feature `train` của fold dùng để train fusion.
- Feature `val` của fold dùng để chọn checkpoint fusion.
- Feature `test` của fold chỉ dùng để báo cáo kết quả cuối.

Đây là cách đúng nếu muốn nói mô hình vẫn speaker-independent.

## Fusion đang ghép những gì?

Với mỗi utterance, notebook nối các tín hiệu sau:

- Embedding của pretrained/raw-audio backbone 03A.
- Emotion probability của 03A.
- VAD prediction của 03A.
- Embedding của co-attention model 03B.
- Emotion probability của 03B.
- VAD prediction của 03B.

Sau đó một MLP fusion học hai head giống các notebook trước: emotion classification và VAD regression.

## Các chỉ số emotion classification

Giả sử có \(K\) lớp emotion và ma trận nhầm lẫn \(C\), trong đó:

$$
C_{ij}
=
\text{số mẫu thuộc lớp thật } i \text{ nhưng dự đoán là lớp } j
$$

### WA / Accuracy / WAR

Trong single-label classification, **WA** thường là accuracy tổng thể:

$$
\operatorname{WA}
=
\frac{\sum_{k=1}^{K} C_{kk}}
{\sum_{i=1}^{K}\sum_{j=1}^{K}C_{ij}}
$$

Một số paper gọi **WAR** là weighted average recall. Với single-label classification, weighted recall theo support thường bằng accuracy:

$$
\operatorname{WAR}
=
\sum_{k=1}^{K}
\frac{n_k}{N}
\operatorname{Recall}_k
$$

với:

$$
n_k=\sum_{j=1}^{K}C_{kj}
$$

### UAR / UA

**UAR** là unweighted average recall. Một số paper viết là **UA**.
Nếu thấy ghi `AUR` trong ghi chú hoặc log, thường đó là viết nhầm của `UAR`; trong SER/IEMOCAP, thuật ngữ chuẩn thường là `UA` hoặc `UAR`.

Recall của lớp \(k\):

$$
\operatorname{Recall}_k
=
\frac{C_{kk}}
{\sum_{j=1}^{K}C_{kj}}
$$

UAR:

$$
\operatorname{UAR}
=
\frac{1}{K}
\sum_{k=1}^{K}
\operatorname{Recall}_k
$$

UAR quan trọng với IEMOCAP vì dataset lệch lớp. Nếu model đoán tốt lớp đông nhưng bỏ qua lớp ít, WA có thể nhìn ổn nhưng UAR sẽ thấp.

### Macro-F1

Precision của lớp \(k\):

$$
\operatorname{Precision}_k
=
\frac{C_{kk}}
{\sum_{i=1}^{K}C_{ik}}
$$

F1 của lớp \(k\):

$$
F1_k
=
\frac{2\operatorname{Precision}_k\operatorname{Recall}_k}
{\operatorname{Precision}_k+\operatorname{Recall}_k}
$$

Macro-F1:

$$
\operatorname{MacroF1}
=
\frac{1}{K}
\sum_{k=1}^{K}F1_k
$$

## Các chỉ số VAD regression

IEMOCAP có dimensional labels gồm:

| Dimension | Ý nghĩa |
|---|---|
| Valence | mức tích cực/tiêu cực của cảm xúc |
| Arousal | mức kích hoạt/năng lượng/cường độ cảm xúc |
| Dominance | mức kiểm soát/tự chủ/áp đảo trong biểu đạt |

### CCC

**CCC** là Concordance Correlation Coefficient. Đây là chỉ số phổ biến cho valence/arousal/dominance vì nó kiểm tra cả tương quan lẫn độ lệch thang đo.

Với ground truth \(y\) và prediction \(\hat{y}\):

$$
\rho_c
=
\frac{2\sigma_{y\hat{y}}}
{\sigma_y^2+\sigma_{\hat{y}}^2+(\mu_y-\mu_{\hat{y}})^2}
$$

Trong đó:

- \(\mu_y\): trung bình ground truth.
- \(\mu_{\hat{y}}\): trung bình prediction.
- \(\sigma_y^2\): phương sai ground truth.
- \(\sigma_{\hat{y}}^2\): phương sai prediction.
- \(\sigma_{y\hat{y}}\): covariance giữa ground truth và prediction.

CCC cao khi prediction vừa tăng/giảm cùng ground truth, vừa không lệch trung bình và độ phân tán quá nhiều.

Loss thường dùng:

$$
\mathcal{L}_{CCC}
=
1-\rho_c
$$

### MAE và RMSE

$$
\operatorname{MAE}
=
\frac{1}{N}
\sum_{i=1}^{N}|y_i-\hat{y}_i|
$$

$$
\operatorname{RMSE}
=
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}(y_i-\hat{y}_i)^2
}
$$

MAE/RMSE dễ hiểu theo thang 1-5, còn CCC dễ so sánh với các paper regression.

## Loss của mô hình 2-head

Emotion classification dùng cross-entropy:

$$
\mathcal{L}_{CE}
=
-
\sum_{k=1}^{K}
y_k\log(\hat{p}_k)
$$

VAD dùng MSE và CCC loss:

$$
\mathcal{L}_{MSE}
=
\frac{1}{3}
\sum_{d\in\{V,A,D\}}
(y_d-\hat{y}_d)^2
$$

$$
\mathcal{L}_{CCC}
=
\frac{1}{3}
\sum_{d\in\{V,A,D\}}
(1-\rho_{c,d})
$$

Tổng loss:

$$
\mathcal{L}
=
\lambda_{CE}\mathcal{L}_{CE}
+
\lambda_{MSE}\mathcal{L}_{MSE}
+
\lambda_{CCC}\mathcal{L}_{CCC}
$$

Trong notebook, các hệ số nằm trong hàm `multitask_loss`. Nếu emotion tốt nhưng CCC thấp, có thể tăng trọng số CCC. Nếu CCC khá nhưng UAR thấp, có thể tăng trọng số CE hoặc xử lý class imbalance.

## Co-attention, cross-attention và bridge tokens

Attention chuẩn:

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt{d_k}}
\right)V
$$

**Self-attention**: \(Q,K,V\) đến từ cùng một nguồn.

**Cross-attention**: \(Q\) đến từ nguồn A, còn \(K,V\) đến từ nguồn B. Ví dụ audio query nhìn vào text key/value.

**Co-attention**: hai nguồn feature tương tác hai chiều hoặc cùng được đưa vào một attention block để học mức liên quan giữa chúng.

**Bridge tokens** trong paper multi-task/multimodal là các vector học được đóng vai trò trung gian giữa hai modality. Thay vì để audio query text trực tiếp, bridge tokens học cách gom thông tin cảm xúc từ audio và text.

Notebook 04 không train bridge-token transformer từ đầu. Nó là bước fusion nhẹ hơn: lấy embedding/prediction đã học từ 03A và 03B, rồi học một MLP 2-head trên các tín hiệu đó.

## Bảng tham chiếu paper trên IEMOCAP

Các kết quả dưới đây không luôn so sánh trực tiếp 1-1 vì mỗi paper có thể khác số mẫu, split, modality và label mapping. Ta dùng bảng này để biết mức điểm kỳ vọng và cách đọc metric.

| Paper | Input/modality | Split | Kiến trúc chính | Kết quả được báo cáo | Ghi chú cho project |
|---|---|---|---|---|---|
| [Antoniou et al., 2023 - Reality check with IEMOCAP](https://arxiv.org/abs/2304.00860) | Review/evaluation guideline | Khuyến nghị 10-fold: 8 speakers train, 1 validation, 1 test | Không phải model mới; phân tích reproducibility | Khuyến nghị baseline tối thiểu dùng 4 lớp `neutral/sad/angry/happy+excited`, khoảng 5531 samples | Dùng để giải thích vì sao 10-fold speaker-independent là setup khó và đáng tin |
| [Chen & Rudnicky, 2021 - Wav2Vec2 fine-tuning](https://arxiv.org/abs/2110.06309) | Audio only/raw waveform | 5-fold leave-one-session-out | wav2vec2 + average pooling + linear classifier; V-FT/TAPT/P-TAPT | UA: V-FT 69.9, TAPT 73.5, P-TAPT 74.3 | Tham chiếu trực tiếp cho 03A: pretrained raw-audio fine-tuning |
| [Ispas et al., 2024 - Multi-task, Multi-modal categorical + dimensional emotion](https://arxiv.org/abs/2401.00536) | Audio + transcript | 10-fold speaker guideline | HuBERT-large + DeBERTaV3-large, self-attention, cross-attention, learnable bridge tokens, multi-task heads | Best table: UAR 74.68, WAR 74.69, Valence CCC .738, Arousal CCC .685 | Tham chiếu chính cho 04: multi-task + cross-attention + 2-head evaluation |
| [emotion2vec, 2023](https://arxiv.org/abs/2312.15185) | Audio representation | IEMOCAP SER | self-supervised emotion representation; downstream linear layers | Paper nhấn mạnh chỉ cần train linear layers đã vượt nhiều pretrained/emotion specialist models | Tham chiếu cho nhánh representation emotion2vec/03B |
| [Sun et al., 2023 - wav2vec2 + BERT auxiliary tasks](https://arxiv.org/abs/2302.13661) | Audio + text | IEMOCAP | wav2vec2 + BERT, fusion bằng multi-head attention, auxiliary tasks | WA 78.42, UA 79.71 | Mốc multimodal cao; project của mình không dùng text chính nên không lấy làm baseline trực tiếp |
| [DST - Deformable Speech Transformer](https://arxiv.org/abs/2302.13729) | Audio/acoustic | IEMOCAP/MELD | transformer với deformable/window attention để học cue đa mức thời gian | Paper báo DST vượt nhiều mô hình transformer SER | Gợi ý cải tiến tương lai nếu muốn thay co-attention bằng temporal attention mạnh hơn |

Khi so sánh, cần ghi rõ:

- 5-fold session hay 10-fold speaker.
- Audio-only hay multimodal audio+text.
- Có dùng transcript/gold text không.
- Có gộp excited vào happy không.
- Dùng WA/UAR/Macro-F1 hay CCC/MAE/RMSE.

## Cách đọc kết quả của notebook 04

Sau khi chạy xong, đọc theo thứ tự:

1. `04_fusion_results_by_fold.csv`: kết quả từng fold.
2. `04_fusion_summary.csv`: mean/std theo `5fold_session` và `10fold_speaker`.
3. So sánh với output của 03A và 03B.

Diễn giải nhanh:

```text
Nếu 04 > 03A và 04 > 03B:
    fusion thật sự có lợi.

Nếu 04 gần 03A nhưng > 03B:
    pretrained backbone là nguồn chính, co-attention chỉ bổ sung nhẹ.

Nếu 04 < 03A:
    fusion đang thêm nhiễu hoặc overfit, cần giảm hidden_dim/dropout hoặc chỉ dùng embedding thay vì prediction.

Nếu UAR tăng nhưng WA giảm:
    model đang công bằng hơn giữa các lớp nhỏ, nhưng mất một ít accuracy tổng.

Nếu CCC_valence thấp hơn CCC_arousal/dominance:
    đây là hiện tượng thường gặp trong speech-only VAD vì valence hay cần lexical/text context hơn arousal.
```

In [ ]:
import os
import sys
import time
import json
import math
import random
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except Exception:
    plt = None
    sns = None

warnings.filterwarnings(
    "ignore",
    message="Support for mismatched key_padding_mask and attn_mask is deprecated.*",
    category=UserWarning,
)

SEED = int(os.getenv("SEED", "42"))

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_DATA_PARALLEL = os.getenv("USE_DATA_PARALLEL", "1") == "1" and N_GPUS > 1
USE_AMP = DEVICE.type == "cuda"

EMOTION_ID_TO_NAME = {0: "neutral", 1: "angry", 2: "sad", 3: "happy"}
NUM_CLASSES = 4

print("Python:", sys.version)
print("Device:", DEVICE)
print("GPU count:", N_GPUS)
print("Use DataParallel:", USE_DATA_PARALLEL)

In [ ]:
LOCAL_PROJECT = Path(r"D:\UTE\Speech Programming\Speech Project")

def unique_existing(paths):
    out = []
    seen = set()
    for item in paths:
        if not item:
            continue
        path = Path(item)
        if path.exists():
            key = str(path.resolve()).lower()
            if key not in seen:
                seen.add(key)
                out.append(path.resolve())
    return out

def search_roots():
    roots = [
        os.getenv("IEMOCAP_DATA_DIR"),
        os.getenv("KAGGLE_INPUT_DIR"),
        Path.cwd(),
        Path.cwd().parent,
        LOCAL_PROJECT / "06_w2v_based_models" / "data",
        LOCAL_PROJECT / "06_w2v_based_models",
        "/kaggle/input",
        "/kaggle/working",
    ]
    return unique_existing(roots)

def find_named_file(filename, env_var=None):
    if env_var and os.getenv(env_var):
        candidate = Path(os.getenv(env_var))
        if candidate.exists():
            return candidate.resolve()
    candidates = []
    for root in search_roots():
        candidates.extend([
            root / filename,
            root / "data" / filename,
            root / "splits" / filename,
            root / "features" / filename,
            root / "metadata" / filename,
            root / "output" / filename,
            root / "output" / "splits" / filename,
            root / "output" / "features" / filename,
            root / "output" / "metadata" / filename,
        ])
        try:
            candidates.extend(root.rglob(filename))
        except Exception:
            pass
    existing = [p.resolve() for p in candidates if p.exists() and p.is_file()]
    if existing:
        return sorted(set(existing), key=lambda p: (len(p.parts), str(p).lower()))[0]
    roots_text = "\n".join(f"- {root}" for root in search_roots())
    raise FileNotFoundError(f"Không tìm thấy {filename}. Notebook đã quét:\n{roots_text}")

def find_audio_dir():
    if os.getenv("IEMOCAP_AUDIO_DIR"):
        candidate = Path(os.getenv("IEMOCAP_AUDIO_DIR"))
        if candidate.exists():
            return candidate.resolve()
    for root in search_roots():
        for candidate in [
            root / "audio_wav",
            root / "data" / "audio_wav",
            root / "output" / "audio_wav",
            root / "datasets" / "AbstractTTS_IEMOCAP" / "audio_wav",
        ]:
            if candidate.exists() and any(candidate.glob("*.wav")):
                return candidate.resolve()
        try:
            for candidate in root.rglob("audio_wav"):
                if candidate.is_dir() and any(candidate.glob("*.wav")):
                    return candidate.resolve()
        except Exception:
            pass
    roots_text = "\n".join(f"- {root}" for root in search_roots())
    raise FileNotFoundError(f"Không tìm thấy thư mục audio_wav. Notebook đã quét:\n{roots_text}")

SPLIT_5FOLD_PATH = find_named_file("iemocap_5fold_session_long.csv", env_var="IEMOCAP_5FOLD_SPLIT_PATH")
SPLIT_10FOLD_PATH = find_named_file("iemocap_10fold_speaker_long.csv", env_var="IEMOCAP_10FOLD_SPLIT_PATH")

def fold_sort_key(name):
    import re
    match = re.search(r"fold_(\d+)", str(name))
    return int(match.group(1)) if match else str(name)

def load_split_table(path, protocol):
    df = pd.read_csv(path)
    required = {"utterance_id", "speaker_id", "session", "emotion_4class", "emotion_id", "valence", "arousal", "dominance", "wav_path", "fold", "split", "train_sample_id"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Thiếu cột trong split {protocol}: {missing}")
    df = df.copy()
    df["protocol"] = protocol
    split_map = {
        "train": "train",
        "training": "train",
        "val": "val",
        "valid": "val",
        "validation": "val",
        "dev": "val",
        "test": "test",
        "testing": "test",
    }
    original_split_values = sorted(df["split"].astype(str).str.lower().str.strip().unique().tolist())
    df["split_original"] = df["split"].astype(str)
    df["split"] = df["split"].astype(str).str.lower().str.strip().map(split_map).fillna(df["split"].astype(str).str.lower().str.strip())
    normalized_split_values = sorted(df["split"].unique().tolist())
    print(f"{protocol} split labels:", original_split_values, "->", normalized_split_values)
    df["emotion_id"] = df["emotion_id"].astype(int)
    for col in ["valence", "arousal", "dominance"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.dropna(subset=["valence", "arousal", "dominance"]).reset_index(drop=True)

def assert_fold_has_required_splits(protocol, fold_name, fold_df):
    counts = fold_df["split"].value_counts().to_dict()
    missing = [name for name in ["train", "val", "test"] if counts.get(name, 0) == 0]
    if missing:
        raise ValueError(
            f"Fold {protocol}/{fold_name} thiếu split {missing}. "
            f"Số lượng hiện có: {counts}. Hãy kiểm tra cột split trong file CSV."
        )

SPLIT_TABLES = {
    "5fold_session": load_split_table(SPLIT_5FOLD_PATH, "5fold_session"),
    "10fold_speaker": load_split_table(SPLIT_10FOLD_PATH, "10fold_speaker"),
}

for protocol, df in SPLIT_TABLES.items():
    print(protocol, "rows:", len(df), "folds:", df["fold"].nunique())
    display(df.groupby(["fold", "split"]).size().unstack(fill_value=0).head(20))

In [ ]:
def vad_to_0_1(values):
    return np.clip((values.astype(np.float32) - 1.0) / 4.0, 0.0, 1.0)

def vad_from_0_1(values):
    return values.astype(np.float32) * 4.0 + 1.0

def concordance_ccc_torch(pred, target, eps=1e-8):
    pred_mean = pred.mean(dim=0)
    target_mean = target.mean(dim=0)
    pred_var = pred.var(dim=0, unbiased=False)
    target_var = target.var(dim=0, unbiased=False)
    cov = ((pred - pred_mean) * (target - target_mean)).mean(dim=0)
    return (2.0 * cov) / (pred_var + target_var + (pred_mean - target_mean).pow(2) + eps)

def concordance_ccc_np(pred, true, eps=1e-8):
    pred = np.asarray(pred, dtype=np.float64)
    true = np.asarray(true, dtype=np.float64)
    pred_mean = pred.mean(axis=0)
    true_mean = true.mean(axis=0)
    pred_var = pred.var(axis=0)
    true_var = true.var(axis=0)
    cov = ((pred - pred_mean) * (true - true_mean)).mean(axis=0)
    return (2.0 * cov) / (pred_var + true_var + (pred_mean - true_mean) ** 2 + eps)

def compute_metrics(y_true, y_pred, vad_true_01, vad_pred_01):
    vad_true_raw = vad_from_0_1(np.asarray(vad_true_01))
    vad_pred_raw = vad_from_0_1(np.asarray(vad_pred_01))
    ccc = concordance_ccc_np(vad_pred_raw, vad_true_raw)
    mae = np.abs(vad_pred_raw - vad_true_raw).mean(axis=0)
    rmse = np.sqrt(((vad_pred_raw - vad_true_raw) ** 2).mean(axis=0))
    return {
        "WA": float(accuracy_score(y_true, y_pred)),
        "UAR": float(balanced_accuracy_score(y_true, y_pred)),
        "Macro_F1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "Weighted_F1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "CCC_valence": float(ccc[0]),
        "CCC_arousal": float(ccc[1]),
        "CCC_dominance": float(ccc[2]),
        "CCC_mean": float(np.mean(ccc)),
        "MAE_mean": float(np.mean(mae)),
        "RMSE_mean": float(np.mean(rmse)),
    }

def primary_score(metrics):
    return 0.35 * metrics["UAR"] + 0.20 * metrics["WA"] + 0.20 * metrics["Macro_F1"] + 0.25 * metrics["CCC_mean"]

def multitask_loss(outputs, emotion_true, vad_true, ce_weight=1.0, mse_weight=0.35, ccc_weight=0.50):
    ce = F.cross_entropy(outputs["emotion_logits"], emotion_true)
    mse = F.mse_loss(outputs["vad_pred"], vad_true)
    ccc_loss = (1.0 - concordance_ccc_torch(outputs["vad_pred"], vad_true)).mean()
    return ce_weight * ce + mse_weight * mse + ccc_weight * ccc_loss

def module_state_dict(model):
    return model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()

def load_module_state_dict(model, state_dict):
    target = model.module if isinstance(model, nn.DataParallel) else model
    target.load_state_dict(state_dict)

def zip_output(output_dir):
    output_dir = Path(output_dir)
    zip_path = output_dir.with_suffix(".zip")
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in output_dir.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(output_dir.parent))
    print("Saved zip:", zip_path)
    return zip_path

In [ ]:
EPOCHS = int(os.getenv("EPOCHS", "80"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "128"))
LR = float(os.getenv("LR", "1e-3"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "1e-4"))
DROPOUT = float(os.getenv("DROPOUT", "0.35"))
HIDDEN_DIM = int(os.getenv("HIDDEN_DIM", "256"))
PATIENCE = int(os.getenv("PATIENCE", "12"))
MAX_FOLDS = int(os.getenv("MAX_FOLDS", "0"))
RUN_PROTOCOLS = [x.strip() for x in os.getenv("RUN_PROTOCOLS", "5fold_session,10fold_speaker").split(",") if x.strip()]

OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "output_04_fusion")).resolve()
MODEL_DIR = OUTPUT_DIR / "models"
REPORT_DIR = OUTPUT_DIR / "reports"
EXTRACT_DIR = OUTPUT_DIR / "extracted_inputs"
for p in [MODEL_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)

In [ ]:
def maybe_extract_feature_zips():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    keywords = ["03a", "03b", "pretrained", "coattention", "output_03"]
    extracted = []
    for root in search_roots():
        try:
            zip_files = list(root.rglob("*.zip"))
        except Exception:
            zip_files = []
        for zip_path in zip_files:
            name = zip_path.name.lower()
            if not any(key in name for key in keywords):
                continue
            target = EXTRACT_DIR / zip_path.stem
            marker = target / ".extracted"
            if marker.exists():
                extracted.append(target)
                continue
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(target)
            marker.write_text("ok", encoding="utf-8")
            extracted.append(target)
            print("Extracted:", zip_path, "->", target)
    if not extracted:
        print("Không thấy zip output 03A/03B cần giải nén. Notebook sẽ tìm thư mục đã giải nén sẵn.")

maybe_extract_feature_zips()

In [ ]:
def find_feature_file(protocol, fold, split, suffix):
    filename = f"{protocol}_{fold}_{split}_{suffix}.npz"
    return find_named_file(filename)

def load_feature_npz(path):
    data = np.load(path, allow_pickle=True)
    return {
        "utterance_id": data["utterance_id"].astype(str),
        "train_sample_id": data["train_sample_id"].astype(str),
        "embedding": data["embedding"].astype(np.float32),
        "emotion_probs": data["emotion_probs"].astype(np.float32),
        "vad_pred": data["vad_pred"].astype(np.float32),
        "emotion_true": data["emotion_true"].astype(np.int64),
        "vad_true": data["vad_true"].astype(np.float32),
    }

def merge_source_features(pretrained, coattention):
    left = pd.DataFrame({"utterance_id": pretrained["utterance_id"], "idx_pre": np.arange(len(pretrained["utterance_id"]))})
    right = pd.DataFrame({"utterance_id": coattention["utterance_id"], "idx_co": np.arange(len(coattention["utterance_id"]))})
    merged = left.merge(right, on="utterance_id", how="inner")
    if len(merged) != len(left) or len(merged) != len(right):
        print("Warning: số mẫu sau khi merge khác nguồn gốc:", len(left), len(right), len(merged))
    ip = merged["idx_pre"].to_numpy()
    ic = merged["idx_co"].to_numpy()
    features = np.concatenate([
        pretrained["embedding"][ip],
        pretrained["emotion_probs"][ip],
        pretrained["vad_pred"][ip],
        coattention["embedding"][ic],
        coattention["emotion_probs"][ic],
        coattention["vad_pred"][ic],
    ], axis=1).astype(np.float32)
    return {
        "utterance_id": merged["utterance_id"].to_numpy(),
        "features": features,
        "emotion_true": pretrained["emotion_true"][ip],
        "vad_true": pretrained["vad_true"][ip],
    }

In [ ]:
class FusionDataset(Dataset):
    def __init__(self, data, scaler=None, fit_scaler=False):
        self.utterance_id = data["utterance_id"]
        x = data["features"]
        if fit_scaler:
            self.scaler = StandardScaler().fit(x)
        else:
            self.scaler = scaler
        self.x = self.scaler.transform(x).astype(np.float32)
        self.y = data["emotion_true"].astype(np.int64)
        self.vad = data["vad_true"].astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx], dtype=torch.float32),
            "emotion_id": torch.tensor(self.y[idx], dtype=torch.long),
            "vad": torch.tensor(self.vad[idx], dtype=torch.float32),
            "utterance_id": str(self.utterance_id[idx]),
        }

def collate_fusion(batch):
    return {
        "x": torch.stack([b["x"] for b in batch]),
        "emotion_id": torch.stack([b["emotion_id"] for b in batch]),
        "vad": torch.stack([b["vad"] for b in batch]),
        "utterance_id": [b["utterance_id"] for b in batch],
    }

def make_loader(dataset, shuffle=False):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0, collate_fn=collate_fusion)

def to_device(batch):
    return {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

class FusionMultiTaskSER(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.35):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.emotion_head = nn.Linear(hidden_dim, NUM_CLASSES)
        self.vad_head = nn.Sequential(nn.Linear(hidden_dim, 3), nn.Sigmoid())

    def forward(self, x, return_embedding=False):
        emb = self.encoder(x)
        out = {"emotion_logits": self.emotion_head(emb), "vad_pred": self.vad_head(emb)}
        if return_embedding:
            out["embedding"] = emb
        return out

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    if len(loader.dataset) == 0:
        raise ValueError("DataLoader rỗng, không thể evaluate. Hãy kiểm tra feature train/val/test của fold.")
    model.eval()
    y_true, y_pred, vad_true, vad_pred, probs = [], [], [], [], []
    utterances = []
    total_loss, n_batches = 0.0, 0
    for batch in loader:
        batch = to_device(batch)
        outputs = model(batch["x"])
        loss = multitask_loss(outputs, batch["emotion_id"], batch["vad"])
        prob = torch.softmax(outputs["emotion_logits"], dim=-1)
        pred = prob.argmax(dim=-1)
        y_true.extend(batch["emotion_id"].detach().cpu().numpy().tolist())
        y_pred.extend(pred.detach().cpu().numpy().tolist())
        vad_true.append(batch["vad"].detach().cpu().numpy())
        vad_pred.append(outputs["vad_pred"].detach().cpu().numpy())
        probs.append(prob.detach().cpu().numpy())
        utterances.extend(batch["utterance_id"])
        total_loss += float(loss.detach().cpu())
        n_batches += 1
    if not vad_true:
        raise ValueError("Không có batch nào trong DataLoader. Dataset đang rỗng.")
    vad_true = np.concatenate(vad_true)
    vad_pred = np.concatenate(vad_pred)
    probs = np.concatenate(probs)
    metrics = compute_metrics(y_true, y_pred, vad_true, vad_pred)
    metrics["loss"] = total_loss / max(n_batches, 1)
    pred_df = pd.DataFrame({"utterance_id": utterances, "true_emotion_id": y_true, "pred_emotion_id": y_pred})
    for i, name in EMOTION_ID_TO_NAME.items():
        pred_df[f"prob_{name}"] = probs[:, i]
    for j, name in enumerate(["valence", "arousal", "dominance"]):
        pred_df[f"true_{name}"] = vad_from_0_1(vad_true)[:, j]
        pred_df[f"pred_{name}"] = vad_from_0_1(vad_pred)[:, j]
    return metrics, pred_df

In [ ]:
def load_split_features(protocol, fold, split):
    pre = load_feature_npz(find_feature_file(protocol, fold, split, "pretrained_features"))
    co = load_feature_npz(find_feature_file(protocol, fold, split, "coattention_features"))
    return merge_source_features(pre, co)

def train_fold(protocol, fold, seed):
    set_seed(seed)
    fold_df = SPLIT_TABLES[protocol][SPLIT_TABLES[protocol]["fold"] == fold].reset_index(drop=True)
    assert_fold_has_required_splits(protocol, fold, fold_df)
    train_data = load_split_features(protocol, fold, "train")
    val_data = load_split_features(protocol, fold, "val")
    test_data = load_split_features(protocol, fold, "test")
    train_ds = FusionDataset(train_data, fit_scaler=True)
    val_ds = FusionDataset(val_data, scaler=train_ds.scaler)
    test_ds = FusionDataset(test_data, scaler=train_ds.scaler)
    train_loader = make_loader(train_ds, shuffle=True)
    val_loader = make_loader(val_ds, shuffle=False)
    test_loader = make_loader(test_ds, shuffle=False)

    model = FusionMultiTaskSER(train_ds.x.shape[1], HIDDEN_DIM, DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_score, best_epoch, bad_epochs = -1e9, -1, 0
    best_path = MODEL_DIR / protocol / f"{fold}_best.pt"
    best_path.parent.mkdir(parents=True, exist_ok=True)
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for batch in train_loader:
            batch = to_device(batch)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(batch["x"])
            loss = multitask_loss(outputs, batch["emotion_id"], batch["vad"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += float(loss.detach().cpu())
        val_metrics, _ = evaluate(model, val_loader)
        score = primary_score(val_metrics)
        history.append({"protocol": protocol, "fold": fold, "epoch": epoch, "train_loss": running / max(len(train_loader), 1), "val_primary_score": score, **{f"val_{k}": v for k, v in val_metrics.items()}})
        print(f"{protocol} | {fold} | epoch {epoch:03d} | val_UAR={val_metrics['UAR']:.4f} | val_CCC={val_metrics['CCC_mean']:.4f} | score={score:.4f}")
        if score > best_score:
            best_score, best_epoch, bad_epochs = score, epoch, 0
            torch.save({"model_state_dict": model.state_dict(), "input_dim": train_ds.x.shape[1], "best_epoch": best_epoch, "best_val_score": best_score}, best_path)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("Early stopping")
                break

    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model_state_dict"])
    test_metrics, pred_df = evaluate(model, test_loader)
    pred_df.to_csv(REPORT_DIR / f"{protocol}_{fold}_test_predictions.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(history).to_csv(REPORT_DIR / f"{protocol}_{fold}_history.csv", index=False, encoding="utf-8-sig")
    result = {"protocol": protocol, "fold": fold, "best_epoch": best_epoch, "best_val_score": best_score, "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds), **test_metrics}
    print("Test:", {k: result[k] for k in ["WA", "UAR", "Macro_F1", "CCC_mean", "MAE_mean"]})
    return result

In [ ]:
all_results = []
start_all = time.time()
for protocol in RUN_PROTOCOLS:
    df = SPLIT_TABLES[protocol]
    folds = sorted(df["fold"].unique().tolist(), key=fold_sort_key)
    if MAX_FOLDS > 0:
        folds = folds[:MAX_FOLDS]
    for idx, fold in enumerate(folds, start=1):
        all_results.append(train_fold(protocol, fold, SEED + idx))

results_df = pd.DataFrame(all_results)
results_df.to_csv(REPORT_DIR / "04_fusion_results_by_fold.csv", index=False, encoding="utf-8-sig")
display(results_df)
summary = results_df.groupby("protocol")[["WA", "UAR", "Macro_F1", "Weighted_F1", "CCC_valence", "CCC_arousal", "CCC_dominance", "CCC_mean", "MAE_mean", "RMSE_mean"]].agg(["mean", "std"]).round(4)
summary.to_csv(REPORT_DIR / "04_fusion_summary.csv", encoding="utf-8-sig")
display(summary)
print("Total seconds:", round(time.time() - start_all, 2))

## Bảng kết quả gọn kiểu paper

Cell này tạo bảng `mean ± std` để đưa trực tiếp vào báo cáo hoặc slide. Bảng tách theo protocol:

- `5fold_session`: đánh giá theo session.
- `10fold_speaker`: đánh giá theo speaker.

Các metric chính:

- Emotion: WA, UAR, Macro-F1.
- Regression: CCC theo Valence/Arousal/Dominance, CCC trung bình, MAE, RMSE.

In [ ]:
def format_mean_std(mean_value, std_value, scale=100.0, digits=2):
    if pd.isna(std_value):
        return f"{mean_value * scale:.{digits}f}"
    return f"{mean_value * scale:.{digits}f} ± {std_value * scale:.{digits}f}"

def build_paper_style_table(results_df):
    rows = []
    metric_cols = ["WA", "UAR", "Macro_F1", "CCC_valence", "CCC_arousal", "CCC_dominance", "CCC_mean", "MAE_mean", "RMSE_mean"]
    for protocol, group in results_df.groupby("protocol"):
        row = {"Protocol": protocol, "Folds": group["fold"].nunique(), "N test total": int(group["n_test"].sum())}
        for metric in metric_cols:
            scale = 100.0 if metric in ["WA", "UAR", "Macro_F1"] else 1.0
            digits = 2 if scale == 100.0 else 4
            row[metric] = format_mean_std(group[metric].mean(), group[metric].std(), scale=scale, digits=digits)
        rows.append(row)
    return pd.DataFrame(rows)

paper_table = build_paper_style_table(results_df)
paper_table_path = REPORT_DIR / "04_fusion_paper_style_results.csv"
paper_table.to_csv(paper_table_path, index=False, encoding="utf-8-sig")
display(paper_table)
print("Saved:", paper_table_path)

## Bảng so sánh với các mô hình tham chiếu

Bảng này đặt kết quả 04 cạnh các paper thường được dùng để tham chiếu trên IEMOCAP. Khi đưa vào báo cáo, cần ghi rõ là các kết quả không hoàn toàn 1-1 vì khác modality, split, số mẫu và cách gộp label.

In [ ]:
reference_rows = [
    {
        "Model": "Chen & Rudnicky 2021 - wav2vec2 V-FT",
        "Input": "audio waveform",
        "Split": "5-fold session",
        "WA/WAR": "",
        "UAR/UA": "69.90",
        "Macro-F1": "",
        "CCC V": "",
        "CCC A": "",
        "CCC D": "",
        "Notes": "Audio-only fine-tuning baseline",
        "Link": "https://arxiv.org/abs/2110.06309",
    },
    {
        "Model": "Chen & Rudnicky 2021 - wav2vec2 P-TAPT",
        "Input": "audio waveform",
        "Split": "5-fold session",
        "WA/WAR": "",
        "UAR/UA": "74.30",
        "Macro-F1": "",
        "CCC V": "",
        "CCC A": "",
        "CCC D": "",
        "Notes": "Task-adaptive pretraining; strong audio-only reference",
        "Link": "https://arxiv.org/abs/2110.06309",
    },
    {
        "Model": "Ispas et al. 2024 - HuBERT + DeBERTaV3 MTL",
        "Input": "audio + transcript",
        "Split": "10-fold speaker",
        "WA/WAR": "74.69",
        "UAR/UA": "74.68",
        "Macro-F1": "",
        "CCC V": "0.738",
        "CCC A": "0.685",
        "CCC D": "",
        "Notes": "Cross-attention, bridge tokens, categorical + dimensional emotion",
        "Link": "https://arxiv.org/abs/2401.00536",
    },
    {
        "Model": "Sun et al. 2023 - wav2vec2 + BERT + auxiliary tasks",
        "Input": "audio + text",
        "Split": "IEMOCAP",
        "WA/WAR": "78.42",
        "UAR/UA": "79.71",
        "Macro-F1": "",
        "CCC V": "",
        "CCC A": "",
        "CCC D": "",
        "Notes": "Multimodal high reference; not directly audio-only comparable",
        "Link": "https://arxiv.org/abs/2302.13661",
    },
]

our_rows = []
for _, row in paper_table.iterrows():
    our_rows.append({
        "Model": "Proposed 04 Fusion - 03A pretrained + 03B co-attention",
        "Input": "audio-only fusion features",
        "Split": row["Protocol"],
        "WA/WAR": row["WA"],
        "UAR/UA": row["UAR"],
        "Macro-F1": row["Macro_F1"],
        "CCC V": row["CCC_valence"],
        "CCC A": row["CCC_arousal"],
        "CCC D": row["CCC_dominance"],
        "Notes": "Same local fold protocol; train-only fusion",
        "Link": "",
    })

comparison_df = pd.DataFrame(reference_rows + our_rows)
comparison_path = REPORT_DIR / "04_fusion_reference_comparison_table.csv"
comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")
display(comparison_df)
print("Saved:", comparison_path)

## Thống kê train/validation/test theo fold

Bảng này giúp kiểm tra lại protocol khi viết báo cáo. Nếu một fold có validation/test bằng 0 thì kết quả không hợp lệ.

In [ ]:
split_stat_rows = []
for protocol in RUN_PROTOCOLS:
    df = SPLIT_TABLES[protocol]
    for fold, group in df.groupby("fold"):
        counts = group["split"].value_counts().to_dict()
        split_stat_rows.append({
            "protocol": protocol,
            "fold": fold,
            "train": counts.get("train", 0),
            "val": counts.get("val", 0),
            "test": counts.get("test", 0),
            "n_speakers_train": group[group["split"] == "train"]["speaker_id"].nunique(),
            "n_speakers_val": group[group["split"] == "val"]["speaker_id"].nunique(),
            "n_speakers_test": group[group["split"] == "test"]["speaker_id"].nunique(),
        })
split_stats_df = pd.DataFrame(split_stat_rows).sort_values(["protocol", "fold"])
split_stats_path = REPORT_DIR / "04_fusion_split_statistics.csv"
split_stats_df.to_csv(split_stats_path, index=False, encoding="utf-8-sig")
display(split_stats_df)
print("Saved:", split_stats_path)

## Biểu đồ train/validation

Các biểu đồ này mô phỏng cách paper thường trình bày quá trình huấn luyện:

- Training loss theo epoch.
- Validation primary score theo epoch.
- Validation UAR và CCC mean theo epoch.

Nếu validation score tăng rồi giảm, model có dấu hiệu overfit. Nếu train loss giảm nhưng validation không tăng, fusion feature có thể chưa bổ sung thông tin hữu ích hoặc MLP quá lớn.

In [ ]:
def load_history_tables():
    history_files = sorted(REPORT_DIR.glob("*_history.csv"))
    frames = []
    for path in history_files:
        df = pd.read_csv(path)
        df["history_file"] = path.name
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

history_all = load_history_tables()
history_path = REPORT_DIR / "04_fusion_all_history.csv"
history_all.to_csv(history_path, index=False, encoding="utf-8-sig")
display(history_all.head())
print("Saved:", history_path)

if plt is not None and len(history_all):
    for protocol, group in history_all.groupby("protocol"):
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        for fold, fg in group.groupby("fold"):
            axes[0].plot(fg["epoch"], fg["train_loss"], alpha=0.55, label=fold)
            axes[1].plot(fg["epoch"], fg["val_primary_score"], alpha=0.55)
            axes[2].plot(fg["epoch"], fg["val_UAR"], alpha=0.55, linestyle="-")
            axes[2].plot(fg["epoch"], fg["val_CCC_mean"], alpha=0.55, linestyle="--")
        axes[0].set_title(f"{protocol} - train loss")
        axes[1].set_title(f"{protocol} - validation primary score")
        axes[2].set_title(f"{protocol} - validation UAR / CCC mean")
        axes[0].set_xlabel("Epoch")
        axes[1].set_xlabel("Epoch")
        axes[2].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[1].set_ylabel("Score")
        axes[2].set_ylabel("Metric")
        axes[0].grid(alpha=0.25)
        axes[1].grid(alpha=0.25)
        axes[2].grid(alpha=0.25)
        axes[0].legend(fontsize=7, ncol=1)
        fig.tight_layout()
        fig_path = REPORT_DIR / f"04_fusion_{protocol}_training_curves.png"
        fig.savefig(fig_path, dpi=180)
        plt.show()
        print("Saved:", fig_path)
else:
    print("Không thể vẽ training curves vì thiếu matplotlib hoặc history.")

## Biểu đồ tổng hợp metric và confusion matrix

Cell này tạo:

- Bar chart WA/UAR/Macro-F1/CCC mean theo protocol.
- Confusion matrix gộp tất cả fold theo từng protocol.
- Bar chart CCC cho Valence/Arousal/Dominance.

In [ ]:
if plt is not None and len(results_df):
    plot_metrics = ["WA", "UAR", "Macro_F1", "CCC_mean"]
    metric_summary = results_df.groupby("protocol")[plot_metrics].agg(["mean", "std"])
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(plot_metrics))
    width = 0.35
    protocols = list(metric_summary.index)
    for idx, protocol in enumerate(protocols):
        means = [metric_summary.loc[protocol, (m, "mean")] for m in plot_metrics]
        stds = [metric_summary.loc[protocol, (m, "std")] for m in plot_metrics]
        offset = (idx - (len(protocols) - 1) / 2) * width
        ax.bar(x + offset, means, width=width, yerr=stds, capsize=4, label=protocol)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_metrics)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("04 Fusion - paper-style metric summary")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig_path = REPORT_DIR / "04_fusion_metric_summary_bar.png"
    fig.savefig(fig_path, dpi=180)
    plt.show()
    print("Saved:", fig_path)

    ccc_metrics = ["CCC_valence", "CCC_arousal", "CCC_dominance"]
    ccc_summary = results_df.groupby("protocol")[ccc_metrics].agg(["mean", "std"])
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(ccc_metrics))
    for idx, protocol in enumerate(protocols):
        means = [ccc_summary.loc[protocol, (m, "mean")] for m in ccc_metrics]
        stds = [ccc_summary.loc[protocol, (m, "std")] for m in ccc_metrics]
        offset = (idx - (len(protocols) - 1) / 2) * width
        ax.bar(x + offset, means, width=width, yerr=stds, capsize=4, label=protocol)
    ax.set_xticks(x)
    ax.set_xticklabels(["Valence", "Arousal", "Dominance"])
    ax.set_ylim(-0.1, 1)
    ax.set_ylabel("CCC")
    ax.set_title("04 Fusion - CCC by VAD dimension")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig_path = REPORT_DIR / "04_fusion_ccc_by_dimension.png"
    fig.savefig(fig_path, dpi=180)
    plt.show()
    print("Saved:", fig_path)
else:
    print("Không thể vẽ metric summary vì thiếu matplotlib hoặc results_df rỗng.")

def plot_confusion_for_protocol(protocol):
    if plt is None:
        return
    files = sorted(REPORT_DIR.glob(f"{protocol}_*_test_predictions.csv"))
    if not files:
        print("Không có prediction file cho", protocol)
        return
    pred = pd.concat([pd.read_csv(p) for p in files], ignore_index=True)
    cm = confusion_matrix(pred["true_emotion_id"], pred["pred_emotion_id"], labels=[0, 1, 2, 3])
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(6, 5))
    if sns is not None:
        sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[EMOTION_ID_TO_NAME[i] for i in range(4)], yticklabels=[EMOTION_ID_TO_NAME[i] for i in range(4)], ax=ax)
    else:
        im = ax.imshow(cm_norm, cmap="Blues")
        fig.colorbar(im, ax=ax)
        ax.set_xticks(range(4), [EMOTION_ID_TO_NAME[i] for i in range(4)])
        ax.set_yticks(range(4), [EMOTION_ID_TO_NAME[i] for i in range(4)])
        for i in range(4):
            for j in range(4):
                ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"04 Fusion normalized confusion matrix - {protocol}")
    fig.tight_layout()
    fig_path = REPORT_DIR / f"04_fusion_{protocol}_confusion_matrix.png"
    fig.savefig(fig_path, dpi=180)
    plt.show()
    print("Saved:", fig_path)

for protocol in RUN_PROTOCOLS:
    plot_confusion_for_protocol(protocol)

## Xuất báo cáo markdown tóm tắt

Cell này tạo một file markdown gom bảng kết quả, bảng so sánh và danh sách hình đã sinh ra. File này tiện để đưa vào báo cáo cuối kỳ hoặc copy sang README.

In [ ]:
report_md = REPORT_DIR / "04_fusion_paper_style_report.md"
figure_files = sorted(REPORT_DIR.glob("04_fusion_*.png"))
with report_md.open("w", encoding="utf-8") as f:
    f.write("# 04 Fusion Paper-Style Report\n\n")
    f.write("## Result table\n\n")
    f.write(paper_table.to_markdown(index=False))
    f.write("\n\n## Reference comparison\n\n")
    f.write(comparison_df.to_markdown(index=False))
    f.write("\n\n## Split statistics\n\n")
    f.write(split_stats_df.to_markdown(index=False))
    f.write("\n\n## Figures\n\n")
    for fig_path in figure_files:
        f.write(f"- {fig_path.name}\n")
print("Saved:", report_md)

In [ ]:
config = {
    "notebook": "04 fusion pretrained backbone + co-attention",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "run_protocols": RUN_PROTOCOLS,
}
(OUTPUT_DIR / "04_run_config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
zip_output(OUTPUT_DIR)